# Download Logs — Episode Replay Processing

Loads raw episode JSON files from `11-download_logs/00-raw`, extracts the winner's observations and actions at each timestep, and saves the processed data to `11-download_logs/01-winner`.

In [18]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import math

CENTER_X = 50.0
CENTER_Y = 50.0
MAX_SPEED = 6.0
PLANET_MARGIN = 0.1

## 01 - Save Winner Data

`get_winner_data(data)` finds the winning agent via `np.argmax(data["rewards"])`, then collects each timestep's observation dict and attaches the agent's action to it.

In [2]:
def get_winner_data(data):
    winner_i = np.argmax(data["rewards"])
    nb_timestep = len(data["steps"])
    new_data = []
    for i in range(nb_timestep):
        obs = data["steps"][i][winner_i]["observation"]
        obs["action"] = data["steps"][i][winner_i]["action"]
        new_data.append(obs)
    return new_data

Iterates over every raw episode file, transforms it with `get_winner_data`, and writes the winner's step list to `01-winner/` under the same filename.

In [3]:
raw_dir = Path("11-download_logs/00-raw")

winner_dir = Path("11-download_logs/01-winner")
winner_dir.mkdir(parents=True, exist_ok=True)

for json_file in sorted(raw_dir.glob("*.json")):
    with open(json_file, "r") as f:
        data = json.load(f)
    new_data = get_winner_data(data)
    out_path = winner_dir / json_file.name
    with open(out_path, "w") as f:
        json.dump(new_data, f)
    print(f"Saved: {out_path}")

Saved: 11-download_logs\01-winner\episode-76303112.json
Saved: 11-download_logs\01-winner\episode-76319029.json


### Sanity Check

Verify that in episode `76319029`:
- `bowwowforeach` is the winner
- At game step 41, a fleet was sent from planet 0 with 70 ships
- That fleet was assigned id 32

**Be careful with the step numbers:**
 - the visualiser starts at step 1 whereas the data starts at step 0   
 - the actions displayed at step *t* has been decided at *t-1*
 - the fleet spawn at planet position at *t-1* plus the radius plus a PLANET_MARGIN = 0.1 toward the angle direction

In [4]:
ep_id = "76319029"

with open(f"11-download_logs/00-raw/episode-{ep_id}.json") as f:
    raw = json.load(f)

winner_i = int(np.argmax(raw["rewards"]))
print(f"Winner: {raw['info']['TeamNames'][winner_i]}")

with open(f"11-download_logs/01-winner/episode-{ep_id}.json") as f:
    winner_data = json.load(f)

# Action at game step 42
step_42 = next(obs for obs in winner_data if obs["step"] == 42)
print(f"Action at step 42: {step_42['action']}")
print(f"Planet 0 at step 42: {step_42['planets'][0]}")

# First appearance of fleet 32
print(f"Fleet 32: ")
for obs in winner_data:
    fleet_32 = [f for f in obs["fleets"] if f[0] == 32]
    if fleet_32:
        print(f" {obs["step"]}: {fleet_32}")

Winner: bowwowforeach
Action at step 42: [[0, 5.199833393096924, 70]]
Planet 0 at step 42: [0, 0, 90.0848389632002, 81.13886629663472, 2.6094379124341005, 5, 5]
Fleet 32: 
 42: [[32, 0, 92.95177753556223, 75.73067016848269, 5.199833393096924, 0, 70]]
 43: [[32, 0, 94.5496980949142, 72.71635103883983, 5.199833393096924, 0, 70]]
 44: [[32, 0, 96.14761865426618, 69.70203190919698, 5.199833393096924, 0, 70]]


# 02 - Augment actions with destination planet, discard useless ships


In [6]:
ep_id = "76319029"


with open(f"11-download_logs/01-winner/episode-{ep_id}.json") as f:
    data = json.load(f)

In [9]:
data
fleet_rows = []
planet_rows = []

for obs in winner_data:
    step = obs["step"]

    for f in obs["fleets"]:
        fleet_rows.append({
            "step":           step,
            "id":             f[0],
            "owner":          f[1],
            "x":              f[2],
            "y":              f[3],
            "angle":          f[4],
            "from_planet_id": f[5],
            "ships":          f[6],
        })

    for p in obs["planets"]:
        planet_rows.append({
            "step":       step,
            "id":         p[0],
            "owner":      p[1],
            "x":          p[2],
            "y":          p[3],
            "radius":     p[4],
            "ships":      p[5],
            "production": p[6],
        })

df_fleet   = pd.DataFrame(fleet_rows).drop_duplicates(subset=["step", "id"]).reset_index(drop=True)
df_planets = pd.DataFrame(planet_rows).drop_duplicates(subset=["step", "id"]).reset_index(drop=True)

print(df_fleet.shape, df_planets.shape)
df_fleet

(1529, 8) (6364, 8)


,step,id,owner,x,y,angle,from_planet_id,ships
0,5,0,0,65.667698,92.864106,4.198040,4,18
1,5,1,1,34.957666,6.702306,0.872665,7,18
2,6,0,0,64.509946,90.815290,4.198040,4,18
3,6,1,1,36.470340,8.505041,0.872665,7,18
4,7,0,0,63.352195,88.766473,4.198040,4,18
...,...,...,...,...,...,...,...,...
1524,165,405,1,69.375049,21.907480,2.687807,34,38
1525,166,405,1,66.758961,23.183431,2.687807,34,38
1526,167,405,1,64.142872,24.459383,2.687807,34,38
1527,168,405,1,61.526784,25.735334,2.687807,34,38


In [35]:
def fleet_speed(nb_ships):
    if nb_ships <= 1:
        return 1.0
    ratio = math.log(nb_ships) / math.log(1000.0)
    return 1.0 + (MAX_SPEED - 1.0) * max(0.0, min(1.0, ratio)) ** 1.5


df_fleet_last = (
    df_fleet
    .groupby("id")
    .last()
    .reset_index()
    .assign(
        speed = lambda df: df["ships"].apply(fleet_speed),
        dx = lambda df: df["speed"] * np.cos(df["angle"]),
        dy = lambda df: df["speed"] * np.sin(df["angle"]),
        next_step = lambda df: df["step"] + 1,
    )
    .merge(
        df_planets,
        left_on=["next_step"],
        right_on=["step"],
        suffixes=("_fleet", "_planet"),
        how="inner"
    )
    .assign(
        t = lambda df: ((df["dx"] * (df["x_planet"] - df["x_fleet"]) + df["dy"] * (df["y_planet"] - df["y_fleet"])) / df["speed"] ** 2).clip(lower=0, upper=1),
        distance = lambda df: np.sqrt((df["x_fleet"] + df["dx"] * df["t"] - df["x_planet"]) ** 2 + (df["y_fleet"] + df["dy"] * df["t"] - df["y_planet"]) ** 2),
        on_target = lambda df: df["distance"] < df["radius"] + PLANET_MARGIN
    )
    .query("on_target")
    .groupby("id_fleet")
    .first()
)
df_fleet_last

,step_fleet,owner_fleet,x_fleet,y_fleet,angle,from_planet_id,ships_fleet,speed,dx,dy,...,id_planet,owner_planet,x_planet,y_planet,radius,ships_planet,production,t,distance,on_target
id_fleet,,,,,,,,,,,,,,,,,,,,,
0,8,0,62.194444,86.717657,4.198040,4,18,2.353303,-1.157751,-2.048816,...,16,0,59.949864,83.857010,2.609438,1,5,1.000000,1.356564,True
1,8,1,39.495689,12.110510,0.872665,7,18,2.353303,1.512674,1.802735,...,19,1,40.050136,16.142990,2.609438,1,5,1.000000,2.426924,True
2,20,1,14.024073,16.798642,2.513274,7,12,2.078772,-1.681762,1.221872,...,3,-1,9.915161,18.861134,2.609438,9,5,1.000000,2.568599,True
3,24,1,13.196892,20.723217,2.967060,19,11,2.022610,-1.991882,0.351222,...,3,1,9.915161,18.861134,2.609438,2,5,1.000000,2.561725,True
4,21,0,37.516949,95.539013,2.608074,16,26,2.619610,-2.255544,1.332246,...,9,0,33.559416,97.876543,2.098612,19,3,1.000000,1.976705,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,160,0,50.569973,24.614537,0.609885,12,44,3.027328,2.481543,1.733971,...,24,0,52.476137,25.670407,1.000000,82,1,0.715906,0.226288,True
409,160,0,53.215182,24.221953,2.257489,16,25,2.590453,-1.642304,2.003319,...,24,0,52.476137,25.670407,1.000000,82,1,0.613290,0.346757,True
410,159,0,92.402453,14.075495,5.061773,22,137,4.005451,1.371142,-3.763457,...,34,0,93.908517,9.941708,1.000000,136,1,1.000000,0.394141,True


In [38]:
fleet_arrival = df_fleet_last["id_planet"].to_dict()

In [39]:
data

[{'angular_velocity': 0.025071608687644256,
  'comet_planet_ids': [],
  'comets': [],
  'fleets': [],
  'initial_planets': [[0,
    -1,
    90.0848389632002,
    81.13886629663472,
    2.6094379124341005,
    21,
    5],
   [1, -1, 18.861133703365283, 90.0848389632002, 2.6094379124341005, 21, 5],
   [2, -1, 81.13886629663472, 9.915161036799802, 2.6094379124341005, 21, 5],
   [3, -1, 9.915161036799802, 18.861133703365283, 2.6094379124341005, 21, 5],
   [4, -1, 67.70762086826534, 96.47405967843181, 1.6931471805599454, 86, 2],
   [5, -1, 3.5259403215681857, 67.70762086826534, 1.6931471805599454, 86, 2],
   [6, -1, 96.47405967843181, 32.29237913173466, 1.6931471805599454, 86, 2],
   [7, -1, 32.29237913173466, 3.5259403215681857, 1.6931471805599454, 86, 2],
   [8, -1, 97.87654348197489, 66.44058367172182, 2.09861228866811, 7, 3],
   [9, -1, 33.559416328278175, 97.87654348197489, 2.09861228866811, 7, 3],
   [10, -1, 66.44058367172182, 2.123456518025108, 2.09861228866811, 7, 3],
   [11, -1, 2

In [40]:
for step, obs in enumerate(data):
    if step == 0:
        continue
    actions = obs["action"]
    new_actions = []
    for action in actions:
        id_fleet = action[0]
        if id_fleet in fleet_arrival:
            new_action = action + [fleet_arrival[id_fleet]]
            new_actions.append(new_action)
    data[step-1]["action"] = new_actions